In [2]:
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker
import math
import pandas as pd
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
from matplotlib.colors import ListedColormap

In [3]:
### ----- SET GLOBAL MATPLOTLIB PARAMS -----

dpi = 300

lrgtxt = 11
medtxt = 9
smltxt = 7

plt.rcParams['xtick.labelsize']=smltxt
plt.rcParams['ytick.labelsize']=smltxt

plt.rcParams['axes.titlesize']=lrgtxt
plt.rcParams['axes.labelsize']=medtxt
plt.rcParams["axes.titlepad"]=12
plt.rcParams['axes.linewidth']=1
plt.rcParams["axes.spines.right"]=False
plt.rcParams["axes.spines.top"]=False

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

plt.rcParams['svg.fonttype'] = 'none'

plt.rcParams['savefig.dpi'] = dpi
plt.rcParams['figure.dpi']= dpi
plt.rcParams['figure.constrained_layout.use'] = True

plt.rcParams['savefig.transparent'] = True

# colors
acols = [[0.627451,   0.57254905, 0.37254903], # all colors
    [0.9607843,  0.7882353,  0.15294118],
    # [0.34901962, 0.35686275, 0.49019608],
    [0.19140625, 0.1953125, 0.265625],
    [0.24705882, 0.30588236, 0.9607843 ]]

fcols = acols[0:2] # female colors, WT first
mcols = acols[2:] # male colors, WT first

# whisker dimensions
whiskspacer = 0.15
whiskpos = -whiskspacer

# plotting parameters
medlw = 1.5 # median linewidth
boxalpha = 0.5 # transparency for boxplot face color
caps = False # don't put caps on the error bars
bwidth = 0.9 # box width
ppad = 0.0085*2 # size of pvalue offset and height of bracket legs
plw = 0.8 # line thickness for pvalue bracket
dotsz = 5

# stats parameters
palpha = 0.05

# programatically set positions
dotdist = 0.15
errdist = 0.15
psdist = 1
gtdist = 1.5
start = 0

# set labels
xlab = ''
ylab = 'percent of assay'
lbls = ['+/+','-/+']

# stats display parameters
pvpct = 0.2

# output directory for figure files
# outdir = 'M:\\scn2a-paper-GWJSNH\\manuscript_v2\\figure_panels'
outdir = 'C:\\Users\\nhogl\\Documents\\Manoli Lab\\2026_scn2a_manuscript\\figure_panels'

# save figures?
savefigs = True

# print stats?
printstats = True

In [4]:
# ----- FUNCTIONS -----

def calculate_transition_matrix(data,normaxis,use='prev_behav'):
    
    # use data to extract transitions
    data['next_behav'] = data.groupby('pair_tag').behavior.shift(periods=-1) # generate match of behavior to next behavior
    data['prev_behav'] = data.groupby('pair_tag').behavior.shift(periods=1) # generate match of behavior to previous behavior
    transitions = data.groupby(['behavior', use]) # organize by unique behavior transitions with key specified by user
    counts = {i[0]:len(i[1]) for i in transitions} # count up instances of transitions
    
    # generate behavior x behavior matrix
    behavs = sorted(data.behavior.unique())
    matrix = pd.DataFrame()

    for x in behavs: # count up transition numbers
        matrix[x] = pd.Series([counts.get((x,y), 0) for y in behavs], index=behavs)  # fill in columns of matrix with previous or next behaviors        
        
    cols = matrix.columns
    
    # calculate probabilities
    matrix[cols] = 100*matrix[cols].div(matrix[cols].sum(axis=normaxis), axis=normaxis) # normalize to 1 across the columns
    
    return matrix

In [5]:
# ----- LOAD DATA -----
data_orig = pd.read_csv(os.path.join('..','all_annotations.csv')).drop(labels=['Unnamed: 0','id'],axis=1)

In [6]:
# ----- FILTER DATA -----

# filter to males
sex = 'M'
data = data_orig[data_orig.sex==sex].copy().reset_index(drop=True)

udata = data[data.assay=='aggression'].copy().reset_index(drop=True)

wt = calculate_transition_matrix(udata[udata.GT=='WT'].copy().reset_index(drop=True),1)
het = calculate_transition_matrix(udata[udata.GT=='Het'].copy().reset_index(drop=True),1)

# integrate wt and het huddle columns and plot in pie charts

In [10]:
wtnext = calculate_transition_matrix(udata[udata.GT=='WT'].copy().reset_index(drop=True),1,use='next_behav')
hetnext = calculate_transition_matrix(udata[udata.GT=='Het'].copy().reset_index(drop=True),1,use='next_behav')

In [17]:
wt

,Aggression receipt,Defensive rear,Defensive strike,Huddle,Investigate,Mount,No interaction,Sniff,Strike,Tussle
Aggression receipt,50.119904,35.820896,56.097561,17.948718,17.094017,0.000000,2.950820,22.038567,15.686275,17.241379
Defensive rear,2.398082,0.000000,7.317073,0.000000,0.854701,0.000000,7.377049,0.000000,5.228758,0.000000
Defensive strike,3.357314,5.970149,7.317073,2.564103,0.854701,0.000000,0.819672,2.754821,1.307190,0.000000
Huddle,0.239808,1.492537,2.439024,0.000000,0.000000,33.333333,11.967213,0.000000,0.653595,0.000000
Investigate,0.239808,1.492537,2.439024,15.384615,0.000000,0.000000,14.918033,1.928375,1.960784,3.448276
Mount,0.000000,0.000000,0.000000,1.282051,0.000000,0.000000,0.327869,0.000000,0.000000,0.000000
No interaction,36.211031,46.268657,19.512195,51.282051,65.811966,0.000000,0.000000,67.493113,33.333333,24.137931
Sniff,1.199041,0.000000,0.000000,11.538462,10.256410,66.666667,53.114754,0.000000,3.921569,17.241379
Strike,7.673861,8.955224,4.878049,0.000000,5.128205,0.000000,2.622951,5.785124,38.562092,37.931034
Tussle,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.754098,0.000000,0.000000,0.000000


In [ ]:
test = pd.concat()

In [20]:
# ----- TRANSITION SCRATCH -----

udata['next_behav'] = udata.groupby('pair_tag').behavior.shift(periods=-1) # generate match of behavior to next behavior
udata['prev_behav'] = udata.groupby('pair_tag').behavior.shift(periods=1) # generate match of behavior to previous behavior
transitions = udata.groupby(['behavior', 'prev_behav']) # organize by unique behavior transitions
counts = {i[0]:len(i[1]) for i in transitions} # count up instances of transitions

# generate behavior x behavior matrix
behavs = sorted(udata.behavior.unique())
matrix = pd.DataFrame()

for x in behavs: # count up transition numbers
    matrix[x] = pd.Series([counts.get((x,y), 0) for y in behavs], index=behavs)
    
cols = matrix.columns

# calculate probabilities across row
normaxis=1
matrix[cols] = matrix[cols].div(matrix[cols].sum(axis=normaxis), axis=normaxis)
# column of matrix is behavior of interest
# each row is the previous behavior

                    Aggression receipt
Aggression receipt                 209
Defensive rear                      10
Defensive strike                    14
Huddle                               1
Investigate                          1
Mount                                0
No interaction                     151
Sniff                                5
Strike                              32
Tussle                               0
                    Aggression receipt  Defensive rear
Aggression receipt                 209              24
Defensive rear                      10               0
Defensive strike                    14               4
Huddle                               1               1
Investigate                          1               1
Mount                                0               0
No interaction                     151              31
Sniff                                5               0
Strike                              32               6
Tussle               

In [19]:
matrix

,Aggression receipt,Defensive rear,Defensive strike,Huddle,Investigate,Mount,No interaction,Sniff,Strike,Tussle
Aggression receipt,0.501199,0.358209,0.560976,0.179487,0.170940,0.000000,0.029508,0.220386,0.156863,0.172414
Defensive rear,0.023981,0.000000,0.073171,0.000000,0.008547,0.000000,0.073770,0.000000,0.052288,0.000000
Defensive strike,0.033573,0.059701,0.073171,0.025641,0.008547,0.000000,0.008197,0.027548,0.013072,0.000000
Huddle,0.002398,0.014925,0.024390,0.000000,0.000000,0.333333,0.119672,0.000000,0.006536,0.000000
Investigate,0.002398,0.014925,0.024390,0.153846,0.000000,0.000000,0.149180,0.019284,0.019608,0.034483
Mount,0.000000,0.000000,0.000000,0.012821,0.000000,0.000000,0.003279,0.000000,0.000000,0.000000
No interaction,0.362110,0.462687,0.195122,0.512821,0.658120,0.000000,0.000000,0.674931,0.333333,0.241379
Sniff,0.011990,0.000000,0.000000,0.115385,0.102564,0.666667,0.531148,0.000000,0.039216,0.172414
Strike,0.076739,0.089552,0.048780,0.000000,0.051282,0.000000,0.026230,0.057851,0.385621,0.379310
Tussle,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.047541,0.000000,0.000000,0.000000


In [21]:
counts

{('Aggression receipt', 'Aggression receipt'): 209,
 ('Aggression receipt', 'Defensive rear'): 10,
 ('Aggression receipt', 'Defensive strike'): 14,
 ('Aggression receipt', 'Huddle'): 1,
 ('Aggression receipt', 'Investigate'): 1,
 ('Aggression receipt', 'No interaction'): 151,
 ('Aggression receipt', 'Sniff'): 5,
 ('Aggression receipt', 'Strike'): 32,
 ('Defensive rear', 'Aggression receipt'): 24,
 ('Defensive rear', 'Defensive strike'): 4,
 ('Defensive rear', 'Huddle'): 1,
 ('Defensive rear', 'Investigate'): 1,
 ('Defensive rear', 'No interaction'): 31,
 ('Defensive rear', 'Strike'): 6,
 ('Defensive strike', 'Aggression receipt'): 23,
 ('Defensive strike', 'Defensive rear'): 3,
 ('Defensive strike', 'Defensive strike'): 3,
 ('Defensive strike', 'Huddle'): 1,
 ('Defensive strike', 'Investigate'): 1,
 ('Defensive strike', 'No interaction'): 8,
 ('Defensive strike', 'Strike'): 2,
 ('Huddle', 'Aggression receipt'): 14,
 ('Huddle', 'Defensive strike'): 2,
 ('Huddle', 'Investigate'): 12,
 ('

In [ ]:
# of all the things that precede a huddle, do the ratios shift from WT to het?
# look at preceding behavior of each huddle